# Run All Experiments (Notebook)

**Workflow:**
1. Run "Imports" and "Configuration" cells once
2. Run "Load Data" cell once (slow, tokenization)
3. Edit strategy/experiment config and re-run "Run Experiments" cell as needed — data stays in memory, strategies are reloaded from disk each time

In [1]:
## Imports (run once)

from __future__ import annotations

import importlib
import multiprocessing as mp
import sys
import time
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

_ROOT = Path(".").resolve().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.config import DEFAULT_TOKENIZER_NAME, RESULTS_DIR
from src.model_config import DEFAULT_MODEL, ModelConfig
from experiments.runner import (
    capacity_from_spec,
    effective_page_size,
    persist_result_row,
    prepare_requests,
    run_simulation,
)
from viz.tree_logger import TreeLogger, _log_filename

print(f"Project root: {_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Default model: {DEFAULT_MODEL.name}  (available: {', '.join(ModelConfig.list_models())})")

Project root: /data/howarli/dev/LLM-prefix-caching-simulator
Results dir:  /data/howarli/dev/LLM-prefix-caching-simulator/results
Default model: qwen3.5-27b  (available: jamba-1.5-mini, llama-3.1-70b, llama-3.1-8b, qwen3-0.6b, qwen3.5-27b, transformer-28l-4096d)


## Configuration

Edit these and re-run. The "Load Data" cell uses `DATASETS`, `ORDERING`, `TOKENIZER`, `SEED`, `MAX_REQUESTS`, `TOKENIZE_WORKERS`.

In [2]:
# ── Data loading config ─────────────────────────────────────────────
DATASETS        = ["swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]
ORDERING        = "timestamp"
TOKENIZER       = DEFAULT_TOKENIZER_NAME
SEED            = 0
MAX_REQUESTS    = 5000
TOKENIZE_WORKERS = 90

# ── Model config ───────────────────────────────────────────────────
MODEL_NAME      = DEFAULT_MODEL.name  # see ModelConfig.list_models()

# ── Logging ─────────────────────────────────────────────────────────
ENABLE_LOG      = False
LOG_DIR         = "viz/logs"

# ── Parallelism ─────────────────────────────────────────────────────
MAX_WORKERS     = 100

## Load Data (run once)

Tokenizes all datasets and caches the results in `PREPARED`. Re-run this cell only if you change `DATASETS`, `ORDERING`, `MAX_REQUESTS`, etc.

In [3]:
# PREPARED: dict mapping (dataset, ordering, tokenizer, max_requests, seed) -> List[TokenizedRequest]
PREPARED: Dict[tuple, list] = {}

for ds in DATASETS:
    prep_key = (ds, ORDERING, TOKENIZER, MAX_REQUESTS, SEED)
    if prep_key in PREPARED:
        print(f"  [skip] {ds}/{ORDERING} already loaded ({len(PREPARED[prep_key])} requests)")
        continue
    t0 = time.perf_counter()
    print(f"  Preparing: {ds}/{ORDERING} (max_requests={MAX_REQUESTS}) ...", end=" ", flush=True)
    reqs = prepare_requests(
        ds, ORDERING, TOKENIZER,
        seed=SEED,
        tokenize_workers=TOKENIZE_WORKERS,
        max_requests=MAX_REQUESTS,
    )
    elapsed = time.perf_counter() - t0
    PREPARED[prep_key] = reqs
    print(f"{len(reqs)} requests ready ({elapsed:.1f}s)")

print(f"\nTotal prep keys loaded: {len(PREPARED)}")

  Preparing: swesmith/timestamp (max_requests=5000) ... 

Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

5000 requests ready (49.3s)
  Preparing: loogle/timestamp (max_requests=5000) ... 1951 requests ready (7.9s)
  Preparing: narrativeqa/timestamp (max_requests=5000) ... 

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

1494 requests ready (24.6s)
  Preparing: sharegpt_90k_raw/timestamp (max_requests=5000) ... 5000 requests ready (14.6s)

Total prep keys loaded: 4


## Run Experiments

Edit the `EXPERIMENTS` list below and re-run this cell. Strategy modules are **reloaded from disk** each time so you can modify strategy code without restarting the kernel or reloading data.

In [7]:
# ── Reload all modules from disk ────────────────────────────────────
# This picks up any code changes without restarting the kernel.
import src.model_config
import src.radix_tree
import src.cache_simulator
import src.metrics
import src.strategies.base
import src.strategies.lru
import src.strategies.lfu
import src.strategies.fifo
import src.strategies.marconi
import src.strategies.marconi2
import src.strategies.marconi3
import src.strategies.crf_decoupling
import src.strategies
import experiments.runner

for mod in [
    src.model_config,
    src.radix_tree,
    src.cache_simulator,
    src.metrics,
    src.strategies.base,
    src.strategies.lru,
    src.strategies.lfu,
    src.strategies.fifo,
    src.strategies.marconi,
    src.strategies.marconi2,
    src.strategies.marconi3,
    src.strategies.crf_decoupling,
    src.strategies,
    experiments.runner,
]:
    importlib.reload(mod)

from experiments.runner import strategy_from_name, capacity_from_spec, run_simulation, persist_result_row, effective_page_size
from src.model_config import ModelConfig
print("All modules reloaded.")

All modules reloaded.


In [ ]:
# ── Define experiments ──────────────────────────────────────────────
# Edit this list freely and re-run from the reload cell above.

EXPERIMENTS: list[dict] = []

for ds in ["swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]:
# for ds in ["sharegpt_90k_raw"]:
    # for page_size in [1, 32, 256, 1024]:
    for page_size in [32]:
        for capacity in [10, 20, 40, 80, 120, 160, "inf"]:
        # for capacity in [10, 20]:
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi2", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3", capacity=capacity))
            EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev0_mn0", capacity=capacity))
            EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev1_mn0", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev2_mn0", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev3_mn0", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev3_mn1", capacity=capacity))

print(f"Total experiments: {len(EXPERIMENTS)}")

Total experiments: 28


In [9]:
# ── Run all experiments ─────────────────────────────────────────────

def _build_strategy_name(cfg: dict) -> str:
    name = cfg["strategy"]
    n = name.lower()
    if n in ("marconi", "marconi2") and cfg.get("marconi_alpha") is not None:
        return f"{name}_{cfg['marconi_alpha']}"
    if n == "crf_decoupling" and cfg.get("crf_lambda_decay") is not None:
        return f"{name}_{cfg['crf_lambda_decay']}"
    return name

def _merge_defaults(cfg: dict) -> dict:
    return {
        "dataset":                cfg.get("dataset", DATASETS[0]),
        "page_size":              cfg.get("page_size", 32),
        "ordering":               cfg.get("ordering", ORDERING),
        "strategy":               cfg.get("strategy", "lru"),
        "capacity":               cfg.get("capacity", "160"),
        "model_name":             cfg.get("model_name", MODEL_NAME),
        "tokenizer":              cfg.get("tokenizer", TOKENIZER),
        "seed":                   cfg.get("seed", SEED),
        "max_requests":           cfg.get("max_requests", MAX_REQUESTS),
        "marconi_alpha":          cfg.get("marconi_alpha"),
        "crf_lambda_decay":       cfg.get("crf_lambda_decay"),
        "crf_c_attn":             cfg.get("crf_c_attn"),
        "crf_c_ssm":              cfg.get("crf_c_ssm"),
    }

def _label(cfg: dict) -> str:
    return f"{cfg['dataset']} ps={cfg['page_size']} {cfg['ordering']} {cfg['strategy']} cap={cfg['capacity']}"

def _sim_worker(args):
    """Run one simulation inside a pool worker (fork inherits PREPARED)."""
    prep_key, page_size, strategy_name, cap, cfg = args
    reqs = PREPARED[prep_key]
    model = ModelConfig.from_name(cfg["model_name"])
    strategy = strategy_from_name(strategy_name, model=model)

    logger = None
    if ENABLE_LOG:
        log_dir = Path(LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        fname = _log_filename(
            dataset=cfg["dataset"], strategy=strategy_name,
            page_size=page_size, capacity_spec=str(cfg["capacity"]),
            ordering=cfg["ordering"], mamba_equiv=model.mamba_state_token_equiv,
        )
        logger = TreeLogger(log_dir / fname)

    try:
        metrics = run_simulation(reqs, page_size, strategy, cap, logger=logger, model=model)
    finally:
        if logger is not None:
            logger.close()

    return {"cfg": cfg, "metrics": metrics.to_dict()}


# Build simulation args
merged = [_merge_defaults(cfg) for cfg in EXPERIMENTS]
sim_args = []
for cfg in merged:
    ds = cfg["dataset"]
    model = ModelConfig.from_name(cfg["model_name"])
    prep_key = (ds, cfg["ordering"], cfg["tokenizer"], cfg["max_requests"], cfg["seed"])
    if prep_key not in PREPARED:
        print(f"  [WARNING] Data not loaded for {prep_key}, skipping")
        continue
    page_size = effective_page_size(ds, int(cfg["page_size"]))
    cap = capacity_from_spec(str(cfg["capacity"]), model=model)
    strategy_name = _build_strategy_name(cfg)
    sim_args.append((prep_key, page_size, strategy_name, cap, cfg))

print(f"Running {len(sim_args)} simulation(s) (model={MODEL_NAME}, max_workers={MAX_WORKERS}) ...")
t_start = time.perf_counter()

ctx = mp.get_context("fork")
done = 0

with ctx.Pool(processes=MAX_WORKERS) as pool:
    for result in pool.imap_unordered(_sim_worker, sim_args):
        cfg = result["cfg"]
        metrics_dict = result["metrics"]

        dataset = cfg["dataset"]
        out_csv = RESULTS_DIR / f"results_{dataset}.csv"
        out_json_dir = RESULTS_DIR / f"json_{dataset}"

        model = ModelConfig.from_name(cfg["model_name"])
        row = {
            "dataset": dataset,
            "page_size": effective_page_size(dataset, int(cfg["page_size"])),
            "ordering": cfg["ordering"],
            "strategy": _build_strategy_name(cfg),
            "capacity_spec": str(cfg["capacity"]),
            "tokenizer": cfg["tokenizer"],
            "mamba_state_token_equiv": model.mamba_state_token_equiv,
            "model_name": cfg["model_name"],
            "metrics": metrics_dict,
        }
        persist_result_row(out_csv, out_json_dir, row)

        done += 1
        token_hr = metrics_dict.get("token_level_hit_rate", 0)
        flops_sr = metrics_dict.get("flops_save_rate")
        flops_str = f"  flops_save={flops_sr:.4f}" if flops_sr is not None else ""
        print(f"  [{done}/{len(sim_args)}] {_label(cfg)}  token_hr={token_hr:.4f}{flops_str}", flush=True)

elapsed = time.perf_counter() - t_start
print(f"\nAll {len(sim_args)} experiments completed in {elapsed:.1f}s.")

Running 28 simulation(s) (model=qwen3.5-27b, max_workers=100) ...
  [1/28] loogle ps=32 timestamp marconi3_ev3_mn1 cap=inf  token_hr=0.9287  flops_save=0.9293
  [2/28] loogle ps=32 timestamp marconi3_ev3_mn1 cap=20  token_hr=0.6107  flops_save=0.6206
  [3/28] loogle ps=32 timestamp marconi3_ev3_mn1 cap=40  token_hr=0.7441  flops_save=0.7525
  [4/28] sharegpt_90k_raw ps=32 timestamp marconi3_ev3_mn1 cap=inf  token_hr=0.9678  flops_save=0.9702
  [5/28] swesmith ps=32 timestamp marconi3_ev3_mn1 cap=inf  token_hr=0.9708  flops_save=0.9715
  [6/28] loogle ps=32 timestamp marconi3_ev3_mn1 cap=10  token_hr=0.4569  flops_save=0.4647
  [7/28] loogle ps=32 timestamp marconi3_ev3_mn1 cap=80  token_hr=0.8889  flops_save=0.8917
  [8/28] loogle ps=32 timestamp marconi3_ev3_mn1 cap=120  token_hr=0.9202  flops_save=0.9214
  [9/28] swesmith ps=32 timestamp marconi3_ev3_mn1 cap=10  token_hr=0.4114  flops_save=0.4164
  [10/28] sharegpt_90k_raw ps=32 timestamp marconi3_ev3_mn1 cap=10  token_hr=0.7311  flo